# Using GPErks

In [ ]:
import numpy as np
import torch
import sys
sys.path.append("/path/to/repos/simulation_toolbox/simulation_toolbox/")
from GPErks_modified.log.logger import get_logger
from GPErks_modified.utils.random import set_seed
from sklearn.model_selection import train_test_split
from GPErks_modified.gp.data.dataset import Dataset
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.means import LinearMean
from gpytorch.kernels import RBFKernel, ScaleKernel
from torchmetrics import MeanSquaredError, R2Score
from GPErks_modified.gp.experiment import GPExperiment
from GPErks_modified.perks.cross_validation import KFoldCrossValidation
from GPErks_modified.train.early_stop import GLEarlyStoppingCriterion
from GPErks_modified.train.emulator import GPEmulator
from GPErks_modified.train.early_stop import NoEarlyStoppingCriterion

log = get_logger()
seed = 8
set_seed(seed)

In [ ]:
import os

# load dataset
mesh = 1
scenario = "51"
basefolder=f"/data/HCM/{mesh}/scenarios/{scenario}"
emulators_folder_base = F"{basefolder}/output/emulators_100fold/"
feature_idx = 0


X_all =  np.loadtxt(f"{basefolder}/data/X.txt", dtype=float)
mask =  np.loadtxt(f"{basefolder}/output/output_mask_beat_5.txt", dtype=float)
mask = mask.astype(bool)


input_masked = X_all[:mask.shape[0]]  # Trim X_all to match the size of the mask
X_ = input_masked[mask]
y_all = np.loadtxt(f"{basefolder}/data/Y.txt", dtype=float)

y_ = y_all[:,feature_idx]

with open(f"{basefolder}/data/xlabels.txt", "r") as f:
        x_labels = f.read().splitlines()

with open(f"{basefolder}/data/ylabels.txt", "r") as f:
        y_labels = f.read().splitlines()

emulators_folder = f"{emulators_folder_base}/{y_labels[feature_idx]}"

X_train = X_
y_train = y_

config_file = f"{emulators_folder}/emulator.ini"


In [ ]:
from GPErks_modified.gp.experiment import load_experiment_from_config_file
from GPErks_modified.serialization.path import posix_path


dataset = Dataset(
                    X_train[:,:13],
                    y_train,
                    x_labels=x_labels[:13],
                    y_label=y_labels[feature_idx]
                    )
                
experiment = load_experiment_from_config_file(
            config_file,
            dataset  # notice that we still need to provide the dataset used!
            )
            
device = "cpu"
            
# loading emulator
best_model_file = posix_path(
    emulators_folder,
    "best_model.pth"
)
best_model_state = torch.load(best_model_file, map_location=torch.device(device))

emul = GPEmulator(experiment, device)
emul.model.load_state_dict(best_model_state)

In [ ]:
# Saltelli method for Sobol' indexes (Si) estimates
from GPErks_modified.perks.gsa import SobolGSA, SobolGSA_ConvexHull, SobolGSA_AlphaShape, SobolGSA_NIMP
# gsa = SobolGSA(dataset=dataset, n=1024, seed=seed)
# gsa = SobolGSA_AlphaShape(dataset=dataset, n=1024, seed=seed, alpha_shape_file=X_train[:,:28],alpha=5)


# estimate Si using the emulator
wave_file = f"/path/to/repos/GSA_for_other_users/fourchamber_GSA/sampling/history_matching/FINAL_ToRORd_Land_HM/wave9/wave_9.json"
gsa = SobolGSA_NIMP(dataset=dataset, n=1024, seed=seed)
gsa.estimate_Sobol_indices_with_emulator(emul, n_draws=10, wave_file=wave_file)
# gsa.summary()

In [ ]:
gsa.correct_Sobol_indices()

In [ ]:
import pandas as pd
df_STi = pd.DataFrame(
    data=np.round(np.median(gsa.ST, axis=0), 6).reshape(-1, 1),
    index=gsa.index_i,
    columns=["STi"],
)
df_Si = pd.DataFrame(
    data=np.round(np.median(gsa.S1, axis=0), 6).reshape(-1, 1),
    index=gsa.index_i,
    columns=["Si"],
)
df_Sij = pd.DataFrame(
    data=np.round(np.median(gsa.S2, axis=0), 6).reshape(-1, 1),
    index=[
        "(" + elem[0] + ", " + elem[1] + ")" for elem in gsa.index_ij
    ],
    columns=["Sij"],
)

# Save df_STi to a CSV file
df_STi.to_csv(f'{emulators_folder}/df_STi.csv', index=True)

# Save df_Si to a CSV file
df_Si.to_csv(f'{emulators_folder}/df_Si.csv', index=True)

# Save df_Sij to a CSV file
df_Sij.to_csv(f'{emulators_folder}/df_Sij.csv', index=True)
